# Tokenizing Texts

Lexos uses the `Tokenizer` module to split text into meaningful units called tokens.  
Unlike the Lexos app, which splits text using simple rules like whitespace, the API uses **spaCy** language models. These models work across many languages and add useful annotations like parts of speech, lemmas, and stop word flags.

This tutorial shows how to use the `Tokenizer` class to process one or more texts with pre-trained language models.



# Language Models

Lexos wraps **spaCy**, a Natural Language Processing library, to load and apply language models for tokenization. These models offer rule-based and statistical approaches, yielding annotated token objects (`spacy.Token`).

> **Important:**  
> While more accurate, language models can consume more memory and may not work well for underrepresented languages.  
> Lexos defaults to `"xx_sent_ud_sm"`—a multilingual model—for broad compatibility.

Each call to `tokenizer.make_doc()` returns a `spacy.Doc` object containing:

- The original text  
- A list of token objects  
- Optional annotations (like `is_punct`, `is_digit`, `pos_`, etc.)

You can also customize the model used by specifying its name (e.g., `"en_core_web_sm"`).


# Setup


In [6]:
from lexos.tokenizer import Tokenizer

#default tokenizer using the multilingual model
tokenizer = Tokenizer()

# specific language model that you can use
tokenizer = Tokenizer(model="en_core_web_sm")

# 📚 Table of Contents

- [Key Terms](#key-terms)
- [Importing Tokenizer](#importing-tokenizer)
- [Loading Data](#loading-data)
- [Tokenizing a Single Text](#tokenizing-a-single-text)
- [Tokenizing Multiple Texts](#tokenizing-multiple-texts)
- [Changing Language Models](#changing-language-models)
- [Working with Ngrams](#working-with-ngrams)
  - [From Text Input](#from-text-input)
  - [From Token List](#from-token-list)
  - [From spaCy Doc](#from-spacy-doc)



## Key Terms

Before proceeding, here are some key terms:

- **Text**: A string of characters before any preprocessing.

- **Token**: A unit of text used in natural language processing (NLP). Most commonly, a token is a word, but it can also be punctuation, a number, or even whitespace — depending on how the text is split.  
  **Example**:  
  Sentence: `"I love NLP!"`  
  Tokens: `['I', 'love', 'NLP', '!']`

- **Document (doc)**: A parsed text returned by the tokenizer. This is a `spacy.Doc` object containing tokens and their attributes.

- **Language Model**: A model that defines how text is segmented into tokens and what annotations are applied.

- **Pipeline**: A series of NLP tasks applied to the text after tokenization (e.g., part-of-speech tagging, dependency parsing).

- **N-gram**: A sequence of *n* tokens used to analyze patterns or context within texts.  
  - Unigrams: single tokens  
  - Bigrams: 2-token sequences → `['I love', 'love NLP']`  
  - Trigrams: 3-token sequences → `['I love NLP']`

- **Stopword**: A commonly used word often filtered out before processing because it adds little semantic value.  
  **Examples**: `['the', 'is', 'and', 'of']`  
  **Use case**: Remove stopwords to focus on more meaningful words.

- **Filtering**: The process of removing certain types of tokens before generating n-grams or running analysis.

  | Term             | Definition                                                                                   | Example / Notes                                         |
  |------------------|----------------------------------------------------------------------------------------------|---------------------------------------------------------|
  | `filter_stops`   | Removes tokens that are stopwords.                                                           | `"This is good"` → `['This', 'good']` (removes `'is'`)  |
  | `filter_punct`   | Removes punctuation-only tokens.                                                             | `"Great!"` → `['Great']`                                |
  | `filter_digits`  | Removes tokens that are only numeric digits.                                                 | `['test', '2023']` → `['test']`                         |
  | `filter_nums`    | Removes tokens that look like numbers, including decimals and formatted numbers.             | `['3.14', '100', 'ten']` → `['ten']`                    |
  | `min_freq`       | Removes n-grams that appear fewer times than a specified threshold.                          | `min_freq=2` removes rare n-grams                       |


## Importing Tokenizer

Import the necessary modules from the Lexos API and ensure the source path is correctly added.

In [ ]:
### DONT ADDD THIS USED FOR IMPORT ERRORS ###

#import sys
#import os

# This adds the absolute path to the src folder directly
#src_path = os.path.abspath(os.path.join(os.getcwd(), '../../'))
#sys.path.insert(0, src_path)

#import sys
#print(sys.version)
#print(sys.executable)




In [7]:
from lexos.tokenizer import Tokenizer

## Loading Data

You can either input text manually or load it from an external file.  
Here, we load in a text file.

To load data to tokenize we'll use the Loader module to load in a text file from Github. The file we'll be using is a small portion of "Pride and Prejudice" by Jane Austen. 

In [ ]:
from lexos.io import loader
loader = loader.Loader()
loader.load(["https://raw.githubusercontent.com/scottkleinman/lexos/refs/heads/main/tests/test_data/txt/Austen_Pride_sm.txt"])
text = loader.texts[0]
text


## Tokenizing a Single Text

Once your text is loaded, you can tokenize it using the `Tokenizer` class.

The recommended method is to use the `make_doc()` function, which takes a string and returns a `spaCy.Doc` object. This object contains the original text and a sequence of annotated tokens.

In [ ]:
tokenizer_def = Tokenizer()
doc = tokenizer_def.make_doc(text)



Tokens:
< >
<Pride>
<and>
<Prejudice>
<
>
<by>
<Jane>
<Austen>
<
>
<Chapter>
<1>
<
>
<It>
<is>
<a>
<truth>
<universally>
<acknowledged>
<,>
<that>
<a>
<single>
<man>
<in>
<possession>
<of>
<a>
<good>
<fortune>
<,>
<must>
<be>
<in>
<want>
<of>
<a>
<wife>
<.>
<
>
<However>
<little>
<known>
<the>
<feelings>
<or>
<views>
<of>
<such>
<a>
<man>


Alternatively, you can call the `Tokenizer` instance directly, just like you would with a `spaCy` `Language` object.  
This automatically routes input to either `make_doc()` or `make_docs()` depending on whether a single string or a list of strings is passed:


In [ ]:
#Alernatively call Tokenizer object directly:
doc = tokenizer_def(text)

After tokenization, you can access the tokens in the returned `Doc` object.  
Here's an example that prints the first 50 tokens:

In [ ]:
print("\nTokens:")
for token in doc[0:50]:
    print(f"<{token.text}>")


> **Note**  
> Some tokens may include punctuation or newline characters. To avoid this, you can either:  
> - Scrub the text using `Scrubber` before tokenization  
> - Filter out unwanted tokens after tokenization using token attributes (e.g., `is_punct`, `is_space`)


## Tokenizing Multiple Texts

The `Tokenizer` class provides the `make_docs()` method to convert a list of raw strings into a list of `spaCy.Doc` objects.  
Each string is tokenized using the language model and returned with full annotations.

> **Note**  
> This method is ideal for batch processing documents.  
> It is functionally similar to calling the `Tokenizer` object directly.


In [ ]:
text_sub1 = text[0:100]
text_sub2 = text[100:200]
text_list = [text_sub1, text_sub2]
docs = list(tokenizer_def.make_docs(text_list))

#Alternatively call Tokenizer object directly:
docs = list(tokenizer_def(text_list))

## Selecting a Model <br>
As mentioned previously, you can select the model that tokenizer uses in order to get more information from or text, or to better fit the language the text is in. In order to do this, you can use the `model` parameter in the `make_doc()` function to override the default model. For this example, we'll use the 'en_web_core_sm' model, since "Pride and Prejudice" is written in english. This model tags parts of speech to each token, as shown below.

In [ ]:
tokenizer_en = Tokenizer(model="en_core_web_sm")
doc = tokenizer_en.make_doc(text)
print("\nTokens with parts of speech:")
for token in doc[0:50]:
    print(f"<{token.text}> : {token.pos_}")

## Ngrams <br>

## Generating Ngrams

Lexos allows you to generate ngrams which are sequences of consecutive tokensfrom either raw text or tokenized documents.  
These are useful for analyzing patterns, building frequency models, or studying frequent phrases in your texts.

Ngrams can be created using:

- A pre-tokenized `spaCy.Doc` object  
- Raw text input  
- Character-based slicing

> **Note**  
> An ngram is a contiguous sequence of *n* items from a given text.  
> For example, 2-grams (bigrams) from `"The end is nigh"` would be:  
> `"The end"`, `"end is"`, `"is nigh"`


In [11]:
from lexos.tokenizer.ngrams import Ngrams

#initialize Ngrams object
ngrams = Ngrams()

### Generating Ngrams from Text

To generate ngrams directly from a raw string (before tokenization), use the `ngrams.from_text()` function.  
This function automatically tokenizes the input and returns ngrams as strings or tokens depending on the `output` parameter.


In [12]:
text = "This is a simple test."
print(list(ngrams.from_text(text, output="text")))
# Output: ['This is', 'is a', 'a simple', 'simple test.']

['This is', 'is a', 'a simple', 'simple test.']


## From Token List <br>


If you already have a list of tokens (e.g. from simple tokenization), you can use from_tokens().

In [ ]:
tokens = ["This", "is", "a", "test"]
print(list(ngrams.from_tokens(tokens, output="tuples")))
# Output: [('This', 'is'), ('is', 'a'), ('a', 'test')]


## From a spaCy Doc <br>
When using spaCy documents (produced by Tokenizer), you can generate spans, tuples, or text.

In [ ]:
doc = tokenizer_def.make_doc("Here is another example.")
print([span.text for span in ngrams.from_doc(doc, output="spans")])
# Output: ['Here is', 'is another', 'another example']


## Filtering Options <br>
The Ngrams class includes filters to clean the text:

- A **filter_stops**: removes stopwords

- A **filter_digits**: removes tokens that are digits

- A **filter_punct**: removes punctuation

In [ ]:
doc = tokenizer_def.make_doc("This is test ten of 10.")
ngrams.filter_digits = True
print(list(ngrams.from_doc(doc, output="text")))
# Output: ['This is', 'is test', 'test ten', 'ten of']


## How to Customize N-Size <Br>

In [ ]:
# Create trigrams (n=3)
ngrams = Ngrams()
doc = tokenizer_def.make_doc("It is a beautiful day outside.")
print(list(ngrams.from_doc(doc, n=3, output="text")))
# Output: ['It is a', 'is a beautiful', 'a beautiful day', 'beautiful day outside']


## Filtering with Multiple Options Combined <Br>

In [ ]:
doc = tokenizer_def.make_doc("This test includes 100%, punctuation, and stopwords.")
ngrams.filter_digits = True
ngrams.filter_punct = True
ngrams.filter_stops = True
print(list(ngrams.from_doc(doc, output="text")))
